# HIG — BreaKHis real-image predictive evaluation

This notebook runs the **real-image application-level experiment** for the HIG paper on the public **BreaKHis** histopathology dataset.

Default protocol:

\[
\text{BreaKHis 40X RGB}
\rightarrow
\text{hematoxylin channel}
\rightarrow
\text{Gaussian smoothing}
\rightarrow
\text{5-level multi-Otsu}
\rightarrow
\text{4-connected constant-label regions}
\rightarrow
\text{HIG/RAG descriptors}.
\]

The same fixed segmentation is used for every representation:

1. segmentation-size baseline;
2. RAG;
3. regional contact multigraph;
4. HIG without cut augmentation;
5. binary-incidence HIG;
6. full HIG.

Prediction is **benign vs malignant** using regularized logistic regression and **patient-grouped nested cross-validation**. No image from the same patient is allowed to appear in both train and test folds.

The notebook exports:

- `breakhis_40x_metadata.csv`
- `breakhis_40x_hig_features.csv`
- `predictive_fold_metrics.csv`
- `oof_predictions.csv`
- `predictive_performance.csv`
- `paired_auc_differences.csv`
- `table_dataset.tex`
- `table_predictive_performance.tex`
- `table_paired_auc.tex`
- `table_hig_class_statistics.tex`
- `breakhis_experimental_section.tex`

## Dataset source

Official BreaKHis page:
`https://web.inf.ufpr.br/vri/databases/breast-cancer-histopathological-database-breakhis/`

The dataset is made available for non-commercial research subject to acknowledgement/citation of the original BreaKHis paper. The archive is about 4.27 GB, so the download and first feature extraction can take some time.

**Important:** the notebook does not hard-code the number of 40X benign/malignant images. It parses and reports the actual archive contents.

In [ ]:
%pip -q install "scikit-image>=0.22" "scikit-learn>=1.4" "numba>=0.59" tqdm

In [ ]:
# ============================================================
# Configuration
# ============================================================

MAGNIFICATION = 40          # main experiment: 40X only
N_LEVELS = 5               # fixed number of segmentation labels
GAUSSIAN_SIGMA = 1.0       # fixed before looking at class labels
MULTIOTSU_NBINS = 64

OUTER_FOLDS = 5
OUTER_REPEATS = 5
INNER_FOLDS = 4
C_GRID = [1e-3, 1e-2, 1e-1, 1.0, 10.0, 100.0]
BOOTSTRAPS = 2000
RANDOM_SEED = 20260927

# Set e.g. MAX_IMAGES=50 for a smoke test. Use None for the final experiment.
MAX_IMAGES = None

import os, sys, re, gc, time, math, json, shutil, tarfile, subprocess
from pathlib import Path

RUNNING_IN_COLAB = "google.colab" in sys.modules

# Repository convention:
#   data/BreakHis_40X/    <- original 40X PNG images (not tracked by git)
#
# Override in any environment with:
#   HIG_BREAKHIS_DATA_DIR=/absolute/path/to/BreakHis_40X
if RUNNING_IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DEFAULT_DATA_DIR = Path(
        "/content/drive/MyDrive/HIG-reproducibility/data/BreakHis_40X"
    )
    WORKDIR = Path(
        "/content/drive/MyDrive/HIG-reproducibility/outputs/breakhis_40x"
    )
else:
    # Local execution from the repository root is the recommended mode.
    REPO_ROOT = Path(os.environ.get("HIG_REPO_ROOT", Path.cwd())).resolve()
    DEFAULT_DATA_DIR = REPO_ROOT / "data" / "BreakHis_40X"
    WORKDIR = REPO_ROOT / "outputs" / "breakhis_40x"

DATA_DIR = Path(
    os.environ.get("HIG_BREAKHIS_DATA_DIR", str(DEFAULT_DATA_DIR))
).expanduser().resolve()

WORKDIR.mkdir(parents=True, exist_ok=True)
ARCHIVE = WORKDIR / "BreaKHis_v1.tar.gz"

OFFICIAL_PAGE = (
    "https://web.inf.ufpr.br/vri/databases/"
    "breast-cancer-histopathological-database-breakhis/"
)

ARCHIVE_URLS = [
    "https://www.inf.ufpr.br/lesoliveira/download/BreaKHis_v1.tar.gz",
    "http://www.inf.ufpr.br/vri/databases/BreaKHis_v1.tar.gz",
]

print("DATA_DIR:", DATA_DIR)
print("DATA_DIR exists:", DATA_DIR.exists())
print("PNG files found:", len(list(DATA_DIR.rglob("*.png"))) if DATA_DIR.exists() else 0)
print("WORKDIR:", WORKDIR)
print("Main magnification:", MAGNIFICATION, "X")


## 1. Download and extract BreaKHis

The download is resumable (`wget -c`). Only files from the selected magnification are extracted, which reduces disk usage. If the archive is already present in `WORKDIR`, it is reused.

In [ ]:
def download_archive():
    if ARCHIVE.exists() and ARCHIVE.stat().st_size > 1_000_000_000:
        print(f"Using existing archive: {ARCHIVE} ({ARCHIVE.stat().st_size/1e9:.2f} GB)")
        return

    last_error = None
    for url in ARCHIVE_URLS:
        print("Trying:", url)
        try:
            subprocess.run(
                ["wget", "-c", "--progress=bar:force:noscroll", "-O", str(ARCHIVE), url],
                check=True,
            )
            if ARCHIVE.stat().st_size > 1_000_000_000:
                print("Download complete.")
                return
        except Exception as exc:
            last_error = exc
            print("Download attempt failed:", exc)

    raise RuntimeError(
        "Automatic download failed. Download BreaKHis_v1.tar.gz from the "
        f"official page and place it at {ARCHIVE}, or place the extracted "
        "40X images under data/BreakHis_40X."
    ) from last_error


def extract_selected_magnification():
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    existing = list(DATA_DIR.rglob("*.png"))
    if existing:
        print(f"Using {len(existing)} already-extracted PNG files from {DATA_DIR}.")
        return

    print(f"Extracting only {MAGNIFICATION}X images...")
    pattern = f"*-{MAGNIFICATION}-*.png"
    cmd = [
        "tar", "-xzf", str(ARCHIVE),
        "-C", str(DATA_DIR),
        "--wildcards", "--no-anchored", pattern
    ]
    try:
        subprocess.run(cmd, check=True)
    except subprocess.CalledProcessError:
        print("GNU-tar wildcard extraction failed; using Python fallback.")
        with tarfile.open(ARCHIVE, "r:gz") as tf:
            root = DATA_DIR.resolve()
            for member in tf:
                if not member.isfile():
                    continue
                name = Path(member.name).name
                if re.search(rf"-{MAGNIFICATION}-\d+\.png$", name, re.I):
                    target = (DATA_DIR / member.name).resolve()
                    if not str(target).startswith(str(root)):
                        raise RuntimeError("Unsafe archive member path.")
                    tf.extract(member, DATA_DIR)

    files = list(DATA_DIR.rglob("*.png"))
    if not files:
        raise RuntimeError("No images were extracted.")
    print(f"Extracted {len(files)} images at {MAGNIFICATION}X.")


def prepare_dataset():
    existing = list(DATA_DIR.rglob("*.png")) if DATA_DIR.exists() else []
    if existing:
        print(f"Using repository data directory: {DATA_DIR}")
        print(f"PNG files found: {len(existing)}")
        return
    print(f"No PNG files found under {DATA_DIR}.")
    print("Falling back to the official archive download.")
    download_archive()
    extract_selected_magnification()


prepare_dataset()


## 2. Parse labels and patient IDs from filenames

BreaKHis filenames encode the tumor class, subtype, patient/slide identifier, magnification and sequence number. We use the parsed patient identifier as the grouping variable for cross-validation.

In [ ]:
import pandas as pd
import numpy as np

FILENAME_RE = re.compile(
    r"^SOB_(?P<class>[BM])_(?P<subtype>A|F|PT|TA|DC|LC|MC|PC)-"
    r"(?P<patient>.+?)-(?P<mag>40|100|200|400)-(?P<seq>\d+)\.png$",
    re.I,
)

def parse_breakhis_file(path):
    m = FILENAME_RE.match(Path(path).name)
    if not m:
        return None
    d = m.groupdict()
    return {
        "image_path": str(Path(path)),
        "filename": Path(path).name,
        "tumor_class": d["class"].upper(),
        "label": 1 if d["class"].upper() == "M" else 0,
        "subtype": d["subtype"].upper(),
        "patient_id": d["patient"],
        "magnification": int(d["mag"]),
        "sequence": int(d["seq"]),
    }

rows = []
unparsed = []
for p in sorted(DATA_DIR.rglob("*.png")):
    d = parse_breakhis_file(p)
    if d is None:
        unparsed.append(str(p))
    elif d["magnification"] == MAGNIFICATION:
        rows.append(d)

meta = pd.DataFrame(rows)
if meta.empty:
    raise RuntimeError("No BreaKHis images were parsed.")

# One patient should have one benign/malignant class.
patient_class_counts = meta.groupby("patient_id")["label"].nunique()
if patient_class_counts.max() != 1:
    raise AssertionError("At least one patient appears in both target classes.")

if MAX_IMAGES is not None:
    # For smoke tests only. Final paper: MAX_IMAGES=None.
    meta = (
        meta.groupby("label", group_keys=False)
        .apply(lambda x: x.sample(
            min(len(x), max(1, MAX_IMAGES // 2)),
            random_state=RANDOM_SEED
        ))
        .reset_index(drop=True)
    )

meta = meta.sort_values(["patient_id", "filename"]).reset_index(drop=True)
META_CSV = WORKDIR / f"breakhis_{MAGNIFICATION}x_metadata.csv"
meta.to_csv(META_CSV, index=False)

print("Parsed images:", len(meta))
print("Patients:", meta["patient_id"].nunique())
print("\nImages by class:")
display(meta.groupby(["tumor_class", "label"]).size().rename("images").reset_index())
print("\nPatients by class:")
display(
    meta.drop_duplicates("patient_id")
        .groupby(["tumor_class", "label"])
        .size().rename("patients").reset_index()
)
if unparsed:
    print(f"Warning: {len(unparsed)} PNG filenames were not parsed.")

## 3. Intrinsic HIG feature extractor

This implementation works on an arbitrary rectangular categorical segmentation.

It uses:

- 4-connected constant-label image regions;
- the pixel-edge frontier graph with `OUT`;
- maximal region-pair frontier continuation, stopped at junctions;
- `seg`, `enc,0`, `enc,1`, `cut`, `junc`, and virtual enclosing generators;
- the canonical regional cut--hole correspondence, computing the number of 4-region / 8-background holes from the digital Euler number;
- exact verification of
  \[
  |V_2|-|V_1|+|V_0|=2.
  \]

The union-find part is JIT-compiled with Numba for speed.

In [ ]:
import scipy.ndimage as ndi
from numba import njit
from skimage.measure import euler_number
from skimage import io, color, filters
from skimage.filters import threshold_multiotsu
from tqdm.auto import tqdm

@njit
def _uf_roots(n, pairs):
    parent = np.arange(n, dtype=np.int64)
    rank = np.zeros(n, dtype=np.int8)

    for k in range(pairs.shape[0]):
        a = pairs[k, 0]
        b = pairs[k, 1]

        ra = a
        while parent[ra] != ra:
            parent[ra] = parent[parent[ra]]
            ra = parent[ra]

        rb = b
        while parent[rb] != rb:
            parent[rb] = parent[parent[rb]]
            rb = parent[rb]

        if ra != rb:
            if rank[ra] < rank[rb]:
                parent[ra] = rb
            elif rank[ra] > rank[rb]:
                parent[rb] = ra
            else:
                parent[rb] = ra
                rank[ra] += 1

    for i0 in range(n):
        i = i0
        r = i
        while parent[r] != r:
            r = parent[r]
        while parent[i] != i:
            p = parent[i]
            parent[i] = r
            i = p

    return parent


def label_4_regions(seg):
    seg = np.asarray(seg)
    structure = np.array([[0,1,0],[1,1,1],[0,1,0]], dtype=bool)
    labels = np.zeros(seg.shape, dtype=np.int32)
    offset = 0

    for value in np.unique(seg):
        lab, n = ndi.label(seg == value, structure=structure)
        mask = lab > 0
        labels[mask] = lab[mask] + offset
        offset += int(n)

    return labels, offset


def segment_histology_rgb(rgb, n_levels=N_LEVELS, sigma=GAUSSIAN_SIGMA):
    rgb = np.asarray(rgb)

    if rgb.ndim == 2:
        gray = rgb.astype(np.float32)
        if gray.max() > 1:
            gray /= 255.0
        scalar = gray
    else:
        if rgb.shape[-1] == 4:
            rgb = color.rgba2rgb(rgb)
        rgbf = rgb.astype(np.float32)
        if rgbf.max() > 1:
            rgbf /= 255.0
        # H&E stain deconvolution; channel 0 is hematoxylin.
        scalar = color.rgb2hed(rgbf)[..., 0]

    if sigma and sigma > 0:
        scalar = filters.gaussian(scalar, sigma=sigma, preserve_range=True)

    try:
        thresholds = threshold_multiotsu(
            scalar, classes=n_levels, nbins=MULTIOTSU_NBINS
        )
    except ValueError:
        # Rare fallback if the image has too few effective levels.
        thresholds = np.unique(
            np.quantile(scalar, np.linspace(0, 1, n_levels + 1)[1:-1])
        )

    seg = np.digitize(scalar, thresholds).astype(np.uint8)
    return seg, scalar, np.asarray(thresholds)


def hig_features_from_segmentation(seg):
    lab1, nreg = label_4_regions(seg)
    R = lab1 - 1
    H, W = R.shape

    # --------------------------------------------------------
    # Frontier edges of the compactified image
    # horizontal edges: (H+1) x W
    # vertical edges:   H x (W+1)
    # --------------------------------------------------------
    h_front = np.zeros((H + 1, W), dtype=bool)
    h_front[0, :] = True
    h_front[H, :] = True
    if H > 1:
        h_front[1:H, :] = R[:-1, :] != R[1:, :]

    v_front = np.zeros((H, W + 1), dtype=bool)
    v_front[:, 0] = True
    v_front[:, W] = True
    if W > 1:
        v_front[:, 1:W] = R[:, :-1] != R[:, 1:]

    Eh = int(h_front.sum())
    Ev = int(v_front.sum())
    E = Eh + Ev

    h_id = np.full(h_front.shape, -1, dtype=np.int64)
    h_id[h_front] = np.arange(Eh, dtype=np.int64)

    v_id = np.full(v_front.shape, -1, dtype=np.int64)
    v_id[v_front] = np.arange(Eh, E, dtype=np.int64)

    edge_a = np.empty(E, dtype=np.int32)
    edge_b = np.empty(E, dtype=np.int32)

    ids = h_id[0, :]
    edge_a[ids] = -1
    edge_b[ids] = R[0, :]

    ids = h_id[H, :]
    edge_a[ids] = -1
    edge_b[ids] = R[-1, :]

    if H > 1:
        m = h_front[1:H, :]
        ids = h_id[1:H, :][m]
        x = R[:-1, :][m]
        y = R[1:, :][m]
        edge_a[ids] = np.minimum(x, y)
        edge_b[ids] = np.maximum(x, y)

    ids = v_id[:, 0]
    edge_a[ids] = -1
    edge_b[ids] = R[:, 0]

    ids = v_id[:, W]
    edge_a[ids] = -1
    edge_b[ids] = R[:, -1]

    if W > 1:
        m = v_front[:, 1:W]
        ids = v_id[:, 1:W][m]
        x = R[:, :-1][m]
        y = R[:, 1:][m]
        edge_a[ids] = np.minimum(x, y)
        edge_b[ids] = np.maximum(x, y)

    lo = np.minimum(edge_a, edge_b) + 1
    hi = np.maximum(edge_a, edge_b) + 1
    pair_key = lo.astype(np.int64) * (nreg + 1) + hi

    # --------------------------------------------------------
    # Frontier junctions
    # --------------------------------------------------------
    degree = np.zeros((H + 1, W + 1), dtype=np.int8)
    degree[:, :-1] += h_front
    degree[:, 1:]  += h_front
    degree[:-1, :] += v_front
    degree[1:, :]  += v_front

    junction_mask = degree >= 3
    n_junc = int(junction_mask.sum())

    # --------------------------------------------------------
    # Maximal same-region-pair continuation through
    # non-junction vertices.
    # --------------------------------------------------------
    sentinel = E
    inc = np.full((4, H + 1, W + 1), sentinel, dtype=np.int64)

    inc[0, :, 1:] = np.where(h_front, h_id, sentinel)
    inc[1, :, :-1] = np.where(h_front, h_id, sentinel)
    inc[2, 1:, :] = np.where(v_front, v_id, sentinel)
    inc[3, :-1, :] = np.where(v_front, v_id, sentinel)

    arr = inc[:, degree == 2].T
    arr.sort(axis=1)

    pairs = arr[:, :2]
    pairs = pairs[pairs[:, 1] < E]
    pairs = pairs[pair_key[pairs[:, 0]] == pair_key[pairs[:, 1]]]

    roots = _uf_roots(E, np.ascontiguousarray(pairs, dtype=np.int64))
    unique_roots, representatives, class_id = np.unique(
        roots, return_index=True, return_inverse=True
    )
    n_geom = len(unique_roots)

    # --------------------------------------------------------
    # Terminal occurrences at junctions
    # --------------------------------------------------------
    vertex_id = np.arange(
        (H + 1) * (W + 1), dtype=np.int64
    ).reshape(H + 1, W + 1)

    eids = []
    vids = []

    tests = [
        (h_front & junction_mask[:, :-1], h_id, vertex_id[:, :-1]),
        (h_front & junction_mask[:, 1:],  h_id, vertex_id[:, 1:]),
        (v_front & junction_mask[:-1, :], v_id, vertex_id[:-1, :]),
        (v_front & junction_mask[1:, :],  v_id, vertex_id[1:, :]),
    ]

    for m, ids_array, vertex_array in tests:
        if np.any(m):
            eids.append(ids_array[m])
            vids.append(vertex_array[m])

    if eids:
        ee = np.concatenate(eids)
        vv = np.concatenate(vids)
        cc = class_id[ee]

        endpoint_count = np.bincount(cc, minlength=n_geom)

        encoded = cc.astype(np.int64) * ((H + 1) * (W + 1)) + vv
        unique_encoded = np.unique(encoded)
        unique_cc = unique_encoded // ((H + 1) * (W + 1))
        unique_junction_count = np.bincount(unique_cc, minlength=n_geom)
    else:
        endpoint_count = np.zeros(n_geom, dtype=np.int64)
        unique_junction_count = np.zeros(n_geom, dtype=np.int64)

    n_enc0 = int(np.sum(endpoint_count == 0))
    n_enc1 = int(np.sum(
        (endpoint_count == 2) & (unique_junction_count == 1)
    ))
    n_seg = int(np.sum(
        (endpoint_count == 2) & (unique_junction_count == 2)
    ))

    valid_class = (
        (endpoint_count == 0)
        | (
            (endpoint_count == 2)
            & ((unique_junction_count == 1) | (unique_junction_count == 2))
        )
    )
    if not np.all(valid_class):
        raise RuntimeError(
            f"{np.count_nonzero(~valid_class)} frontier classes could "
            "not be assigned to seg/enc,0/enc,1."
        )

    # --------------------------------------------------------
    # Regional 4-(8) holes.
    # For each 4-connected region R:
    #   holes(R) = 1 - Euler_4(R).
    # --------------------------------------------------------
    holes = np.zeros(nreg, dtype=np.int32)
    for rid, sl in enumerate(ndi.find_objects(lab1), start=1):
        if sl is None:
            continue
        mask = lab1[sl] == rid
        holes[rid - 1] = max(
            0, 1 - int(euler_number(mask, connectivity=1))
        )

    n_cut = int(holes.sum())

    # --------------------------------------------------------
    # Region contact graph / multigraph.
    # One geometric continuation class represents one contact.
    # --------------------------------------------------------
    class_a = edge_a[representatives]
    class_b = edge_b[representatives]
    image_contact = (class_a >= 0) & (class_b >= 0)

    c_a = class_a[image_contact]
    c_b = class_b[image_contact]
    class_pair_key = (
        np.minimum(c_a, c_b).astype(np.int64) * nreg
        + np.maximum(c_a, c_b)
    )

    if class_pair_key.size:
        _, contact_multiplicity = np.unique(
            class_pair_key, return_counts=True
        )
        contact_n_edges = int(class_pair_key.size)
        contact_max_multiplicity = int(contact_multiplicity.max())
        contact_mean_multiplicity = float(contact_multiplicity.mean())
    else:
        contact_n_edges = 0
        contact_max_multiplicity = 0
        contact_mean_multiplicity = 0.0

    # --------------------------------------------------------
    # RAG from image-image adjacencies, parallel contacts collapsed.
    # --------------------------------------------------------
    rag_keys = []

    if H > 1:
        m = R[:-1, :] != R[1:, :]
        if np.any(m):
            a = R[:-1, :][m]
            b = R[1:, :][m]
            rag_keys.append(
                np.minimum(a, b).astype(np.int64) * nreg
                + np.maximum(a, b)
            )

    if W > 1:
        m = R[:, :-1] != R[:, 1:]
        if np.any(m):
            a = R[:, :-1][m]
            b = R[:, 1:][m]
            rag_keys.append(
                np.minimum(a, b).astype(np.int64) * nreg
                + np.maximum(a, b)
            )

    if rag_keys:
        rag_key = np.unique(np.concatenate(rag_keys))
        rag_n_edges = int(len(rag_key))

        ra = (rag_key // nreg).astype(np.int64)
        rb = (rag_key % nreg).astype(np.int64)
        rag_degree = np.bincount(
            np.concatenate([ra, rb]), minlength=nreg
        )
        rag_max_degree = int(rag_degree.max())
    else:
        rag_n_edges = 0
        rag_max_degree = 0

    rag_mean_degree = 2.0 * rag_n_edges / nreg
    rag_density = (
        2.0 * rag_n_edges / (nreg * (nreg - 1))
        if nreg > 1 else 0.0
    )
    # Quotient adjacency graph of a connected rectangular image is connected.
    rag_cycle_rank = rag_n_edges - nreg + 1

    interior_frontier_length = 0
    if H > 1:
        interior_frontier_length += int(
            np.count_nonzero(R[:-1, :] != R[1:, :])
        )
    if W > 1:
        interior_frontier_length += int(
            np.count_nonzero(R[:, :-1] != R[:, 1:])
        )

    # --------------------------------------------------------
    # HIG counts and aggregate incidence statistics.
    # --------------------------------------------------------
    n_v2 = nreg + 1
    n_v1 = n_geom + n_cut
    n_v0 = n_junc + n_enc0

    chi_cell = n_v2 - n_v1 + n_v0
    if chi_cell != 2:
        raise AssertionError(
            f"HIG Euler consistency failed: chi_cell={chi_cell}"
        )

    # 0-1:
    # seg: two distinct junction supports
    # enc,1: two occurrences at one junction
    # enc,0: multiplicity 2 at one virtual 0-generator
    nnz01 = 2 * n_seg + n_enc1 + n_enc0
    m01 = 2 * n_geom

    # 1-2:
    # geometric generator -> two different 2-generators
    # cut generator -> multiplicity 2 at its carrier
    nnz12 = 2 * n_geom + n_cut
    m12 = 2 * n_v1

    graph_vertices = n_v0 + n_v1 + n_v2
    graph_edges = m01 + m12
    chi_graph = graph_vertices - graph_edges
    beta1_graph = graph_edges - graph_vertices + 1

    # Binary-incidence HIG.
    binary_edges = nnz01 + nnz12
    chi_graph_binary = graph_vertices - binary_edges
    beta1_graph_binary = binary_edges - graph_vertices + 1

    # HIG without cut augmentation.
    n_v1_nocut = n_geom
    graph_vertices_nocut = n_v0 + n_v1_nocut + n_v2
    nnz12_nocut = 2 * n_geom
    m12_nocut = 2 * n_geom
    graph_edges_nocut = m01 + m12_nocut
    chi_graph_nocut = graph_vertices_nocut - graph_edges_nocut
    beta1_graph_nocut = graph_edges_nocut - graph_vertices_nocut + 1

    d01 = nnz01 / (n_v0 * n_v1) if n_v0 and n_v1 else 0.0
    d12 = nnz12 / (n_v1 * n_v2) if n_v1 and n_v2 else 0.0

    d01_nocut = (
        nnz01 / (n_v0 * n_geom)
        if n_v0 and n_geom else 0.0
    )
    d12_nocut = (
        nnz12_nocut / (n_geom * n_v2)
        if n_geom and n_v2 else 0.0
    )

    cubical_cells = 4 * H * W + 2 * H + 2 * W + 1
    rho_hig = graph_vertices / cubical_cells

    return {
        "height": H,
        "width": W,
        "n_pixels": H * W,
        "n_regions": nreg,
        "mean_region_area": (H * W) / nreg,
        "frontier_length": interior_frontier_length,

        "rag_n_vertices": nreg,
        "rag_n_edges": rag_n_edges,
        "rag_mean_degree": rag_mean_degree,
        "rag_max_degree": rag_max_degree,
        "rag_density": rag_density,
        "rag_cycle_rank": rag_cycle_rank,

        "contact_n_edges": contact_n_edges,
        "contact_parallel_excess": contact_n_edges - rag_n_edges,
        "contact_mean_multiplicity": contact_mean_multiplicity,
        "contact_max_multiplicity": contact_max_multiplicity,
        "contact_cycle_rank": contact_n_edges - nreg + 1,

        "n_v2_img": nreg,
        "n_v2": n_v2,

        "n_v1_seg": n_seg,
        "n_v1_enc1": n_enc1,
        "n_v1_enc0": n_enc0,
        "n_v1_cut": n_cut,
        "n_v1": n_v1,

        "n_v0_junc": n_junc,
        "n_v0_enc": n_enc0,
        "n_v0": n_v0,

        "seg_per_region": n_seg / nreg,
        "cut_per_region": n_cut / nreg,
        "junc_per_region": n_junc / nreg,

        "d01": d01,
        "d12": d12,
        "m01": m01,
        "m12": m12,

        "holes_mean": float(holes.mean()),
        "holes_max": int(holes.max()),
        "holes_var": float(holes.var()),
        "holes_prop_positive": float(np.mean(holes > 0)),

        "chi_cell": chi_cell,
        "chi_graph": chi_graph,
        "beta1_graph": beta1_graph,

        "chi_graph_binary": chi_graph_binary,
        "beta1_graph_binary": beta1_graph_binary,

        "n_v1_nocut": n_v1_nocut,
        "d01_nocut": d01_nocut,
        "d12_nocut": d12_nocut,
        "m01_nocut": m01,
        "m12_nocut": m12_nocut,
        "chi_graph_nocut": chi_graph_nocut,
        "beta1_graph_nocut": beta1_graph_nocut,

        "rho_hig": rho_hig,
        "frontier_edges_total": E,
    }


# ------------------------------------------------------------
# Internal regression tests.
# ------------------------------------------------------------

# Three concentric squares.
_test = np.zeros((10, 10), dtype=np.uint8)
_test[1:9, 1:9] = 1
_test[3:7, 3:7] = 2
_test[4:6, 4:6] = 3

f = hig_features_from_segmentation(_test)
assert f["n_regions"] == 4
assert f["n_v1_enc0"] == 4
assert f["n_v1_cut"] == 3
assert f["n_v0_enc"] == 4
assert f["chi_cell"] == 2
assert f["chi_graph"] == -6

# Random consistency stress test.
_rng = np.random.default_rng(12345)
for _ in range(100):
    _seg = _rng.integers(0, 5, size=(10, 10), dtype=np.uint8)
    assert hig_features_from_segmentation(_seg)["chi_cell"] == 2

print("HIG internal tests: PASSED")

## 4. Preview the fixed preprocessing

This cell shows one benign and one malignant image, the hematoxylin scalar image, and the resulting 5-level segmentation. It is a **quality-control visualization**, not a tuning step.

In [ ]:
import matplotlib.pyplot as plt

sample_rows = []
for target in [0, 1]:
    x = meta[meta["label"] == target]
    if len(x):
        sample_rows.append(x.iloc[0])

fig, axes = plt.subplots(len(sample_rows), 3, figsize=(12, 4 * len(sample_rows)))
if len(sample_rows) == 1:
    axes = np.array([axes])

for i, row in enumerate(sample_rows):
    rgb = io.imread(row["image_path"])
    seg, scalar, thr = segment_histology_rgb(rgb)

    axes[i, 0].imshow(rgb)
    axes[i, 0].set_title(
        f"{'Malignant' if row['label'] else 'Benign'} — RGB"
    )
    axes[i, 1].imshow(scalar, cmap="gray")
    axes[i, 1].set_title("Hematoxylin scalar image")
    axes[i, 2].imshow(seg, cmap="viridis", vmin=0, vmax=N_LEVELS-1)
    axes[i, 2].set_title(
        f"{N_LEVELS}-level segmentation\n"
        + "thresholds=" + np.array2string(thr, precision=3)
    )

    for ax in axes[i]:
        ax.axis("off")

plt.tight_layout()
PREVIEW_PNG = WORKDIR / f"breakhis_{MAGNIFICATION}x_segmentation_preview.png"
plt.savefig(PREVIEW_PNG, dpi=180, bbox_inches="tight")
plt.show()

print("Saved:", PREVIEW_PNG)

## 5. Extract HIG/RAG features

Results are checkpointed every 20 images. If the Colab session stops, rerunning the cell resumes from the existing CSV.

In [ ]:
FEATURE_CSV = WORKDIR / f"breakhis_{MAGNIFICATION}x_hig_features.csv"
ERROR_CSV = WORKDIR / f"breakhis_{MAGNIFICATION}x_errors.csv"

if FEATURE_CSV.exists():
    old = pd.read_csv(FEATURE_CSV)
    done = set(old["filename"].astype(str))
    feature_rows = old.to_dict("records")
    print(f"Resuming from {len(feature_rows)} completed images.")
else:
    done = set()
    feature_rows = []

error_rows = []

pending = meta[~meta["filename"].isin(done)]

for count, row in enumerate(
    tqdm(pending.itertuples(index=False), total=len(pending)),
    start=1
):
    t0 = time.perf_counter()
    try:
        rgb = io.imread(row.image_path)
        seg, scalar, thresholds = segment_histology_rgb(rgb)
        feat = hig_features_from_segmentation(seg)
        feat.update({
            "filename": row.filename,
            "image_path": row.image_path,
            "label": int(row.label),
            "tumor_class": row.tumor_class,
            "subtype": row.subtype,
            "patient_id": row.patient_id,
            "magnification": int(row.magnification),
            "segmentation_levels": int(N_LEVELS),
            "gaussian_sigma": float(GAUSSIAN_SIGMA),
            "runtime_seconds": time.perf_counter() - t0,
        })
        feature_rows.append(feat)
    except Exception as exc:
        error_rows.append({
            "filename": row.filename,
            "image_path": row.image_path,
            "error": repr(exc),
        })

    if count % 20 == 0:
        pd.DataFrame(feature_rows).to_csv(FEATURE_CSV, index=False)
        if error_rows:
            pd.DataFrame(error_rows).to_csv(ERROR_CSV, index=False)
        gc.collect()

features = pd.DataFrame(feature_rows)
features.to_csv(FEATURE_CSV, index=False)

if error_rows:
    pd.DataFrame(error_rows).to_csv(ERROR_CSV, index=False)

print("Completed:", len(features), "/", len(meta))
print("Errors:", len(error_rows))
print("Mean extraction time: %.3f s/image" % features["runtime_seconds"].mean())
print("Euler check:", features["chi_cell"].value_counts().to_dict())

if len(features) != len(meta):
    print("WARNING: final paper analysis should be run only after resolving all extraction errors.")

## 6. Structural quality-control tables

These are descriptive only. They are not used to choose preprocessing parameters.

In [ ]:
if not np.all(features["chi_cell"].to_numpy() == 2):
    raise AssertionError("At least one real-image HIG failed chi_cell = 2.")

display_cols = [
    "n_regions",
    "n_v1_seg",
    "n_v1_enc0",
    "n_v1_enc1",
    "n_v1_cut",
    "n_v0_junc",
    "n_v0_enc",
    "chi_graph",
    "rho_hig",
]

class_stats = (
    features.groupby("tumor_class")[display_cols]
    .agg(["mean", "std", "median"])
)

display(class_stats)

CLASS_STATS_CSV = WORKDIR / "hig_class_statistics.csv"
class_stats.to_csv(CLASS_STATS_CSV)

print("Total cuts processed:", int(features["n_v1_cut"].sum()))
print("All chi_cell = 2:", bool((features["chi_cell"] == 2).all()))

## 7. Nested patient-grouped predictive evaluation

Feature sets are nested so that richer representations inherit low-order descriptors recoverable from the poorer representation.

The target is benign (`0`) versus malignant (`1`). All preprocessing in the classifier (`StandardScaler`) is fitted inside the training fold only.

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
)

BASELINE = [
    "n_regions",
    "frontier_length",
    "mean_region_area",
]

RAG = BASELINE + [
    "rag_n_edges",
    "rag_mean_degree",
    "rag_max_degree",
    "rag_density",
    "rag_cycle_rank",
]

CONTACT = RAG + [
    "contact_n_edges",
    "contact_parallel_excess",
    "contact_mean_multiplicity",
    "contact_max_multiplicity",
    "contact_cycle_rank",
]

HIG_NO_CUT = CONTACT + [
    "n_v1_seg",
    "n_v1_enc0",
    "n_v1_enc1",
    "n_v0_junc",
    "n_v0_enc",
    "seg_per_region",
    "junc_per_region",
    "d01_nocut",
    "d12_nocut",
    "m01_nocut",
    "m12_nocut",
    "chi_graph_nocut",
    "beta1_graph_nocut",
]

BINARY_HIG = CONTACT + [
    "n_v1_seg",
    "n_v1_enc0",
    "n_v1_enc1",
    "n_v1_cut",
    "n_v0_junc",
    "n_v0_enc",
    "seg_per_region",
    "cut_per_region",
    "junc_per_region",
    "holes_mean",
    "holes_max",
    "holes_var",
    "holes_prop_positive",
    "d01",
    "d12",
    "chi_graph_binary",
    "beta1_graph_binary",
]

FULL_HIG = BINARY_HIG + [
    "m01",
    "m12",
    "chi_graph",
    "beta1_graph",
]

FEATURE_SETS = {
    "Segmentation-size baseline": BASELINE,
    "RAG": RAG,
    "Regional contact multigraph": CONTACT,
    "HIG without cuts": HIG_NO_CUT,
    "Binary-incidence HIG": BINARY_HIG,
    "Full HIG": FULL_HIG,
}

# Ensure all requested variables exist and are finite.
for name, cols in FEATURE_SETS.items():
    missing = sorted(set(cols) - set(features.columns))
    if missing:
        raise KeyError(f"{name}: missing features {missing}")

work = features.copy().reset_index(drop=True)
work["sample_index"] = np.arange(len(work))

y = work["label"].to_numpy(dtype=int)
groups = work["patient_id"].astype(str).to_numpy()

# Check that 5 patient-group folds are feasible.
patient_targets = work.drop_duplicates("patient_id")["label"].value_counts()
if patient_targets.min() < OUTER_FOLDS:
    raise RuntimeError(
        f"Not enough patients in the smaller class for {OUTER_FOLDS} grouped folds."
    )

fold_rows = []
oof_rows = []

for rep in range(OUTER_REPEATS):
    outer = StratifiedGroupKFold(
        n_splits=OUTER_FOLDS,
        shuffle=True,
        random_state=RANDOM_SEED + rep,
    )

    outer_splits = list(outer.split(np.zeros(len(work)), y, groups))

    for fold, (train_idx, test_idx) in enumerate(outer_splits):
        y_train = y[train_idx]
        y_test = y[test_idx]
        g_train = groups[train_idx]

        inner_n = min(
            INNER_FOLDS,
            work.iloc[train_idx].drop_duplicates("patient_id")["label"]
                .value_counts().min()
        )
        if inner_n < 2:
            raise RuntimeError("Grouped inner CV is not feasible.")

        inner = StratifiedGroupKFold(
            n_splits=int(inner_n),
            shuffle=True,
            random_state=RANDOM_SEED + 10_000 + rep * 100 + fold,
        )
        inner_splits = list(
            inner.split(
                np.zeros(len(train_idx)),
                y_train,
                g_train,
            )
        )

        for representation, cols in FEATURE_SETS.items():
            X = work[cols].to_numpy(dtype=float)
            X_train = X[train_idx]
            X_test = X[test_idx]

            model = Pipeline([
                ("scale", StandardScaler()),
                ("clf", LogisticRegression(
                    solver="liblinear",
                    class_weight="balanced",
                    max_iter=5000,
                    random_state=RANDOM_SEED,
                )),
            ])

            search = GridSearchCV(
                estimator=model,
                param_grid={"clf__C": C_GRID},
                scoring="roc_auc",
                cv=inner_splits,
                n_jobs=-1,
                refit=True,
            )
            search.fit(X_train, y_train)

            prob = search.predict_proba(X_test)[:, 1]
            pred = (prob >= 0.5).astype(int)

            auc = roc_auc_score(y_test, prob)
            prauc = average_precision_score(y_test, prob)
            ba = balanced_accuracy_score(y_test, pred)

            tn, fp, fn, tp = confusion_matrix(
                y_test, pred, labels=[0, 1]
            ).ravel()
            sensitivity = tp / (tp + fn) if (tp + fn) else np.nan
            specificity = tn / (tn + fp) if (tn + fp) else np.nan

            fold_rows.append({
                "repeat": rep,
                "fold": fold,
                "representation": representation,
                "roc_auc": auc,
                "pr_auc": prauc,
                "balanced_accuracy": ba,
                "sensitivity": sensitivity,
                "specificity": specificity,
                "best_C": search.best_params_["clf__C"],
                "n_test": len(test_idx),
                "n_test_patients": len(np.unique(groups[test_idx])),
            })

            for local_i, idx in enumerate(test_idx):
                oof_rows.append({
                    "repeat": rep,
                    "fold": fold,
                    "sample_index": int(idx),
                    "filename": work.loc[idx, "filename"],
                    "patient_id": work.loc[idx, "patient_id"],
                    "label": int(y[idx]),
                    "representation": representation,
                    "probability": float(prob[local_i]),
                })

fold_metrics = pd.DataFrame(fold_rows)
oof = pd.DataFrame(oof_rows)

FOLD_CSV = WORKDIR / "predictive_fold_metrics.csv"
OOF_CSV = WORKDIR / "oof_predictions.csv"
fold_metrics.to_csv(FOLD_CSV, index=False)
oof.to_csv(OOF_CSV, index=False)

display(
    fold_metrics.groupby("representation")[
        ["roc_auc", "pr_auc", "balanced_accuracy", "sensitivity", "specificity"]
    ].agg(["mean", "std"])
)

## 8. Cross-fitted metrics and patient-level bootstrap confidence intervals

For each image and representation, the repeated out-of-fold probabilities are averaged first. Confidence intervals are then obtained by resampling **patients**, not individual images.

In [ ]:
# Average repeated cross-fitted predictions for each image.
avg_oof = (
    oof.groupby(
        ["sample_index", "filename", "patient_id", "label", "representation"],
        as_index=False
    )["probability"].mean()
)

def metric_vector(y_true, probability):
    pred = (probability >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(
        y_true, pred, labels=[0, 1]
    ).ravel()
    return {
        "roc_auc": roc_auc_score(y_true, probability),
        "pr_auc": average_precision_score(y_true, probability),
        "balanced_accuracy": balanced_accuracy_score(y_true, pred),
        "sensitivity": tp / (tp + fn) if (tp + fn) else np.nan,
        "specificity": tn / (tn + fp) if (tn + fp) else np.nan,
    }

def patient_bootstrap_metrics(df, n_boot=BOOTSTRAPS, seed=RANDOM_SEED):
    rng = np.random.default_rng(seed)
    patient_ids = df["patient_id"].astype(str).unique()
    index_by_patient = {
        p: np.flatnonzero(df["patient_id"].astype(str).to_numpy() == p)
        for p in patient_ids
    }

    y0 = df["label"].to_numpy(dtype=int)
    p0 = df["probability"].to_numpy(dtype=float)

    point = metric_vector(y0, p0)
    boots = {k: [] for k in point}

    attempts = 0
    while len(boots["roc_auc"]) < n_boot and attempts < n_boot * 4:
        attempts += 1
        sampled = rng.choice(patient_ids, size=len(patient_ids), replace=True)
        idx = np.concatenate([index_by_patient[p] for p in sampled])
        yb = y0[idx]
        pb = p0[idx]
        if len(np.unique(yb)) < 2:
            continue
        m = metric_vector(yb, pb)
        for k, v in m.items():
            boots[k].append(v)

    result = {}
    for k, value in point.items():
        vals = np.asarray(boots[k], dtype=float)
        result[k] = value
        result[k + "_lo"] = np.nanpercentile(vals, 2.5)
        result[k + "_hi"] = np.nanpercentile(vals, 97.5)
    return result

performance_rows = []
for representation in FEATURE_SETS:
    df = avg_oof[avg_oof["representation"] == representation].copy()
    m = patient_bootstrap_metrics(
        df,
        seed=RANDOM_SEED + list(FEATURE_SETS).index(representation)
    )
    m["representation"] = representation
    performance_rows.append(m)

performance = pd.DataFrame(performance_rows)
PERFORMANCE_CSV = WORKDIR / "predictive_performance.csv"
performance.to_csv(PERFORMANCE_CSV, index=False)

display(performance)

## 9. Paired AUC differences

The principal comparisons isolate:

\[
\mathrm{contact\ multigraph}-\mathrm{RAG},
\]

\[
\mathrm{full\ HIG}-\mathrm{HIG\ without\ cuts},
\]

and

\[
\mathrm{full\ HIG}-\mathrm{binary\ HIG}.
\]

The confidence interval is obtained from the same patient bootstrap sample for both representations.

In [ ]:
wide = avg_oof.pivot_table(
    index=["sample_index", "filename", "patient_id", "label"],
    columns="representation",
    values="probability",
).reset_index()

COMPARISONS = [
    ("Regional contact multigraph", "RAG", "Contact multigraph - RAG"),
    ("Full HIG", "HIG without cuts", "Full HIG - HIG without cuts"),
    ("Full HIG", "Binary-incidence HIG", "Full HIG - Binary HIG"),
    ("Full HIG", "RAG", "Full HIG - RAG"),
]

def paired_auc_bootstrap(df, model_a, model_b, n_boot=BOOTSTRAPS, seed=RANDOM_SEED):
    rng = np.random.default_rng(seed)
    patient_ids = df["patient_id"].astype(str).unique()
    pseries = df["patient_id"].astype(str).to_numpy()
    index_by_patient = {
        p: np.flatnonzero(pseries == p)
        for p in patient_ids
    }

    y = df["label"].to_numpy(dtype=int)
    pa = df[model_a].to_numpy(dtype=float)
    pb = df[model_b].to_numpy(dtype=float)

    point_a = roc_auc_score(y, pa)
    point_b = roc_auc_score(y, pb)
    point = point_a - point_b

    values = []
    attempts = 0
    while len(values) < n_boot and attempts < n_boot * 4:
        attempts += 1
        sampled = rng.choice(patient_ids, size=len(patient_ids), replace=True)
        idx = np.concatenate([index_by_patient[p] for p in sampled])
        yb = y[idx]
        if len(np.unique(yb)) < 2:
            continue
        values.append(
            roc_auc_score(yb, pa[idx]) - roc_auc_score(yb, pb[idx])
        )

    values = np.asarray(values)
    return {
        "auc_a": point_a,
        "auc_b": point_b,
        "delta_auc": point,
        "delta_auc_lo": np.percentile(values, 2.5),
        "delta_auc_hi": np.percentile(values, 97.5),
    }

paired_rows = []
for i, (a, b, label) in enumerate(COMPARISONS):
    d = paired_auc_bootstrap(
        wide, a, b,
        seed=RANDOM_SEED + 1000 + i
    )
    d.update({"comparison": label, "representation_a": a, "representation_b": b})
    paired_rows.append(d)

paired = pd.DataFrame(paired_rows)
PAIRED_CSV = WORKDIR / "paired_auc_differences.csv"
paired.to_csv(PAIRED_CSV, index=False)

display(paired)

## 10. Export publication-ready LaTeX tables and subsection

After the full run, the generated `.tex` files contain **no `XX` placeholders**.

In [ ]:
def ci_text(row, metric):
    return (
        f"{row[metric]:.3f} "
        f"[{row[metric + '_lo']:.3f}, {row[metric + '_hi']:.3f}]"
    )

# ------------------------------------------------------------
# Dataset table
# ------------------------------------------------------------
img_counts = (
    meta.groupby("tumor_class").size()
    .reindex(["B", "M"], fill_value=0)
)
patient_counts = (
    meta.drop_duplicates("patient_id")
        .groupby("tumor_class").size()
        .reindex(["B", "M"], fill_value=0)
)

dataset_tex = rf"""
\begin{{table}}[ht]
\centering
\caption{{BreaKHis {MAGNIFICATION}$\times$ subset used for the
application-level evaluation.}}
\label{{tab:breakhis-data}}
\begin{{tabular}}{{lrr}}
\hline
Class & Images & Patients \\
\hline
Benign & {int(img_counts["B"])} & {int(patient_counts["B"])} \\
Malignant & {int(img_counts["M"])} & {int(patient_counts["M"])} \\
\hline
Total & {len(meta)} & {meta["patient_id"].nunique()} \\
\hline
\end{{tabular}}
\end{{table}}
""".strip()

(WORKDIR / "table_dataset.tex").write_text(dataset_tex, encoding="utf-8")


# ------------------------------------------------------------
# Predictive performance table
# ------------------------------------------------------------
ordered = list(FEATURE_SETS.keys())
pmap = performance.set_index("representation")

lines = []
for representation in ordered:
    row = pmap.loc[representation]
    lines.append(
        f"{representation} & "
        f"{ci_text(row, 'roc_auc')} & "
        f"{ci_text(row, 'pr_auc')} & "
        f"{ci_text(row, 'balanced_accuracy')} \\\\"
    )

perf_tex = r"""
\begin{table}[ht]
\centering
\caption{Patient-grouped cross-fitted predictive performance.
Entries are point estimates with 95\% patient-bootstrap confidence intervals.}
\label{tab:representation-performance}
\begin{tabular}{lccc}
\hline
Representation & ROC--AUC & PR--AUC & Balanced accuracy \\
\hline
""" + "\n".join(lines) + r"""
\hline
\end{tabular}
\end{table}
""".strip()

(WORKDIR / "table_predictive_performance.tex").write_text(
    perf_tex, encoding="utf-8"
)


# ------------------------------------------------------------
# Paired AUC table
# ------------------------------------------------------------
paired_lines = []
for _, row in paired.iterrows():
    paired_lines.append(
        f"{row['comparison']} & "
        f"{row['delta_auc']:+.3f} & "
        f"[{row['delta_auc_lo']:+.3f}, {row['delta_auc_hi']:+.3f}] \\\\"
    )

paired_tex = r"""
\begin{table}[ht]
\centering
\caption{Paired differences in ROC--AUC from the same cross-fitted
predictions. Confidence intervals are obtained by patient-level bootstrap.}
\label{tab:paired-auc}
\begin{tabular}{lcc}
\hline
Comparison & $\Delta$AUC & 95\% CI \\
\hline
""" + "\n".join(paired_lines) + r"""
\hline
\end{tabular}
\end{table}
""".strip()

(WORKDIR / "table_paired_auc.tex").write_text(
    paired_tex, encoding="utf-8"
)


# ------------------------------------------------------------
# Structural class summary table
# ------------------------------------------------------------
summary_vars = [
    ("n_regions", r"$|V_2^{\mathrm{img}}|$"),
    ("n_v1_seg", r"$|V_1^{\mathrm{seg}}|$"),
    ("n_v1_enc0", r"$|V_1^{\mathrm{enc},0}|$"),
    ("n_v1_enc1", r"$|V_1^{\mathrm{enc},1}|$"),
    ("n_v1_cut", r"$|V_1^{\mathrm{cut}}|$"),
    ("n_v0_junc", r"$|V_0^{\mathrm{junc}}|$"),
    ("n_v0_enc", r"$|V_0^{\mathrm{enc}}|$"),
    ("chi_graph", r"$\chi_{\mathrm{graph}}$"),
]

s_lines = []
for col, label in summary_vars:
    b = features.loc[features["label"] == 0, col]
    m = features.loc[features["label"] == 1, col]
    s_lines.append(
        f"{label} & {b.mean():.2f} $\\pm$ {b.std(ddof=1):.2f} & "
        f"{m.mean():.2f} $\\pm$ {m.std(ddof=1):.2f} \\\\"
    )

struct_tex = r"""
\begin{table}[ht]
\centering
\caption{Low-order HIG statistics for benign and malignant
BreaKHis images. Values are mean $\pm$ standard deviation.
These quantities are descriptive and are not univariate hypothesis tests.}
\label{tab:breakhis-hig-statistics}
\begin{tabular}{lcc}
\hline
Quantity & Benign & Malignant \\
\hline
""" + "\n".join(s_lines) + r"""
\hline
\end{tabular}
\end{table}
""".strip()

(WORKDIR / "table_hig_class_statistics.tex").write_text(
    struct_tex, encoding="utf-8"
)


# ------------------------------------------------------------
# Automatically generated manuscript subsection
# ------------------------------------------------------------
full = pmap.loc["Full HIG"]
rag = pmap.loc["RAG"]

def paired_lookup(name):
    return paired.set_index("comparison").loc[name]

d_contact = paired_lookup("Contact multigraph - RAG")
d_cut = paired_lookup("Full HIG - HIG without cuts")
d_mult = paired_lookup("Full HIG - Binary HIG")
d_rag = paired_lookup("Full HIG - RAG")

section_tex = rf"""
\subsection{{Application-level predictive evaluation on BreaKHis}}
\label{{subsec:breakhis-predictive}}

We complement the controlled synthetic experiments with a real-image
classification experiment on the BreaKHis breast histopathology dataset.
The analysis is restricted to the {MAGNIFICATION}$\times$ magnification in
order to avoid confounding the representation comparison with changes in
microscopic scale. The selected subset contains {len(meta)} RGB images from
{meta["patient_id"].nunique()} patients.

\input{{table_dataset.tex}}

Each image is converted to a scalar hematoxylin representation by H\&E
stain deconvolution. A Gaussian filter with fixed standard deviation
$\sigma={GAUSSIAN_SIGMA:g}$ pixels is applied, after which the image is
quantized into {N_LEVELS} levels by multi-Otsu thresholding. The
4-connected components of equal quantized value define the segmentation
used by all compared representations. Neither the segmentation procedure
nor any of its parameters uses the benign/malignant labels.

For every segmentation we compute the segmentation-size baseline, the
ordinary RAG, the regional contact multigraph, the HIG without cut
augmentation, the binary-incidence HIG, and the full HIG. The latter
includes the typed generator counts, normalized generator statistics,
regional-hole descriptors, incidence densities, integer incidence
multiplicities, and graph-level quantities described in
Section~\ref{{subsec:hig-descriptors}}. The cellular quantity
$\chi_{{\mathrm{{cell}}}}$ is excluded from the predictor because it is
identically equal to $2$.

All {len(features)} processed images satisfied the implementation-level
identity
\[
|V_2|-|V_1|+|V_0|=2.
\]
Across this real-image experiment, the intrinsic construction processed
a total of {int(features["n_v2_img"].sum())} image-region generators and
{int(features["n_v1_cut"].sum())} cut generators.

The prediction task is benign versus malignant histology. We use
regularized logistic regression with class-balanced training weights.
Continuous descriptors are standardized from the training data only.
The regularization parameter is selected by grouped inner
cross-validation. The outer evaluation uses
{OUTER_FOLDS}-fold stratified group cross-validation repeated
{OUTER_REPEATS} times, with patient identity as the grouping variable;
consequently, images from one patient never occur simultaneously in
training and test sets.

Repeated out-of-fold probabilities are averaged per image before the
final metrics are computed. Uncertainty is estimated by
{BOOTSTRAPS} patient-level bootstrap resamples.

\input{{table_hig_class_statistics.tex}}

\input{{table_predictive_performance.tex}}

The full HIG obtains a cross-fitted ROC--AUC of
{full["roc_auc"]:.3f}
(95\% CI [{full["roc_auc_lo"]:.3f}, {full["roc_auc_hi"]:.3f}]).
For comparison, the RAG obtains
{rag["roc_auc"]:.3f}
(95\% CI [{rag["roc_auc_lo"]:.3f}, {rag["roc_auc_hi"]:.3f}]).
The paired full-HIG minus RAG difference is
{d_rag["delta_auc"]:+.3f}
(95\% CI [{d_rag["delta_auc_lo"]:+.3f},
{d_rag["delta_auc_hi"]:+.3f}]).

The ablation comparisons are reported in
Table~\ref{{tab:paired-auc}}. The contact-multigraph minus RAG difference
is {d_contact["delta_auc"]:+.3f}
(95\% CI [{d_contact["delta_auc_lo"]:+.3f},
{d_contact["delta_auc_hi"]:+.3f}]); the full-HIG minus
HIG-without-cuts difference is {d_cut["delta_auc"]:+.3f}
(95\% CI [{d_cut["delta_auc_lo"]:+.3f},
{d_cut["delta_auc_hi"]:+.3f}]); and the full-HIG minus
binary-incidence-HIG difference is {d_mult["delta_auc"]:+.3f}
(95\% CI [{d_mult["delta_auc_lo"]:+.3f},
{d_mult["delta_auc_hi"]:+.3f}]).
These paired comparisons are interpreted as representation-level
ablations rather than as tests of the intrinsic mathematical correctness
of the HIG.

\input{{table_paired_auc.tex}}
""".strip()

SECTION_TEX = WORKDIR / "breakhis_experimental_section.tex"
SECTION_TEX.write_text(section_tex, encoding="utf-8")

print("\nGenerated files:")
for p in [
    WORKDIR / "table_dataset.tex",
    WORKDIR / "table_hig_class_statistics.tex",
    WORKDIR / "table_predictive_performance.tex",
    WORKDIR / "table_paired_auc.tex",
    SECTION_TEX,
]:
    print(" -", p)

print("\n--- Generated manuscript subsection ---\n")
print(section_tex)

## 11. Bundle results

Run this only after the complete experiment. It creates one ZIP containing the CSV results, LaTeX tables, generated subsection and segmentation preview.

In [ ]:
import zipfile

BUNDLE = WORKDIR / f"HIG_BreaKHis_{MAGNIFICATION}x_results.zip"

wanted = [
    META_CSV,
    FEATURE_CSV,
    WORKDIR / "predictive_fold_metrics.csv",
    WORKDIR / "oof_predictions.csv",
    WORKDIR / "predictive_performance.csv",
    WORKDIR / "paired_auc_differences.csv",
    WORKDIR / "hig_class_statistics.csv",
    WORKDIR / "table_dataset.tex",
    WORKDIR / "table_hig_class_statistics.tex",
    WORKDIR / "table_predictive_performance.tex",
    WORKDIR / "table_paired_auc.tex",
    WORKDIR / "breakhis_experimental_section.tex",
    PREVIEW_PNG,
]

with zipfile.ZipFile(BUNDLE, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for p in wanted:
        p = Path(p)
        if p.exists():
            zf.write(p, arcname=p.name)

print("Bundle:", BUNDLE)

try:
    from google.colab import files as colab_files
    colab_files.download(str(BUNDLE))
except Exception:
    print("Not running in Colab; download the ZIP manually from WORKDIR.")